<a href="https://colab.research.google.com/github/nandakumar3261/ai-mentor-portfolio-Nandakumar/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

In [3]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [4]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 13.4 MB/s eta 0:00:00


In [6]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

How do you create remarkable change? By hiring, celebrating and nurturing the best people-from all walks of life. We’re here to help! Tell us what you’re looking for and we’ll get you connected to the right people. Discover the latest TCS hiring 2026 updates, including TCS recruitment 2026, upcoming TCS jobs 2026, and all new TCS vacancies 2026 across India. Find complete details on TCS freshers h


In [7]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_682/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [9]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '06f7506c-4f8e-403d-9251-8ac7a573e115', 'type': 'tool_call'}]

[2] ToolMessage
    Content: 1 week ago - TCS NQT 2026 Recruitment is open for 2024–2026 batches. Check eligibility, salary up to ₹12.26 LPA, exam pattern, important dates, and how to apply online. 1 week ago - For 2026, TCS has retained the 60% aggregate threshold introduced in 2022 but tightened its standing arrears policy, c

[3] AIMessage
    Content: [{'type': 'text', 'text': "I'm sorry, but I couldn't find a specific numerical hiring quota for TCS in 2026 in the search results. The information available primarily discusses the TCS NQT hiring process for the 2026 batch, including eligibility and application details, rather than a specific target


In [10]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [11]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))

aws, spring boot


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))

**Score: 2/10**

**Rationale:** Extremely generic, shows no research, and highlights purely self-serving motivations, which is a major red flag in an interview.


In [13]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_682/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [14]:
import json, pathlib
profiles = json.loads(pathlib.Path('/content/sample_data/student_profiles.json').read_text())

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {p["target_company"]}')
    print(f'{"="*70}')

    msg = (f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
           f"skills: {', '.join(p['skills'])}. Target: {p['target_company']}. "
           f"Plan 3 mock interview questions for me, score one of my sample answers, "
           f"and tell me what skills I need to add to be a strong fit.")

    result = agent.invoke({'messages': [('user', msg)]}, config={'recursion_limit': 10})

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')


Student 1: Ravi Kumar — CSE CGPA 8.2 → TCS Digital

  [0] HumanMessage
      I am Ravi Kumar, B.Tech CSE CGPA 8.2, skills: Python, Java, SQL, Git. Target: TCS Digital. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Ravi! It\'s great to hear about your profile. Here are 3 mock interview questions that you might encounter for a TCS Digital role:\n\n1.  **Technical (OOP):** "Explain the concept of Object-Oriented Programming (OOP) and its four main pillars with examples in Java or

Student 2: Sneha Reddy — ECE CGPA 7.6 → Cognizant

  [0] HumanMessage
      I am Sneha Reddy, B.Tech ECE CGPA 7.6, skills: C++, Python, MATLAB, Verilog. Target: Cognizant. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hi Sneha, to help you be

In [23]:
@tool
def jd_fetcher(url: str) -> str:
    """
    Fetch JD text.
    Returns sample JD for demo/testing.
    """

    return """
    TCS Digital Hiring

    Required Skills:
    - Python
    - SQL
    - DBMS
    - Data Structures
    - OOPs
    - Communication Skills
    - Cloud Basics
    """

In [25]:
result = agent.invoke(
    {
        "messages": [
            (
                "user",
                "Fetch this JD and tell me the must-have skills."
            )
        ]
    }
)

for j, m in enumerate(result["messages"]):

    print(f"\n[{j}] {type(m).__name__}")

    if hasattr(m, "content") and m.content:
        print(str(m.content)[:500])

    if hasattr(m, "tool_calls") and m.tool_calls:
        for tc in m.tool_calls:
            print(
                f'→ {tc.get("name")}({tc.get("args")})'
            )


[0] HumanMessage
Fetch this JD and tell me the must-have skills.

[1] AIMessage
Please provide the URL of the Job Description.
